## Varying Pooling Types

This example constructs a classifier for the CIFAR-10 dataset using *Max Pooling*, *Average Pooling* and *Strided Convolutions*.

In [ ]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load and Prepare the Data

In [ ]:
# Define the transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load the dataset into memory
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Function to load data into GPU
def load_data_to_gpu(dataset):
    data_loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    data_iter = iter(data_loader)
    images, labels = next(data_iter)
    return images.to('cuda'), labels.to('cuda')

# Load the train and test sets to GPU
train_images, train_labels = load_data_to_gpu(trainset)
test_images, test_labels = load_data_to_gpu(testset)

# Create TensorDatasets with data on GPU
train_dataset = TensorDataset(train_images, train_labels)
test_dataset = TensorDataset(test_images, test_labels)

# Create DataLoaders with smaller batch sizes for training and testing
trainloader = DataLoader(train_dataset, batch_size=2048, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = batch_y
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = batch_y
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

### Build CNN Model using Max Pooling

The CNN model is constructed using 3 Convolutional layers and 2 Max Pooling layers before finishing with a dense layer before the softmax. This is essentially the same structure as used with MNIST but the input size and number of channels has changed.

In [ ]:
class CNNModelMax(nn.Module):
    def __init__(self):
        super(CNNModelMax, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = x.view(-1, 256) 
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
no_epochs = 200

In [ ]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
history_dict = dict()

In [ ]:
cnn_model_max = CNNModelMax().to(device)
optimizer = optim.Adam(cnn_model_max.parameters(), lr=0.001)

history_dict["Max Pooling"] = train_model(cnn_model_max, trainloader, testloader,
                                                                        loss_fn, optimizer, no_epochs)

### Average Pooling

In [ ]:
class CNNModelAve(nn.Module):
    def __init__(self):
        super(CNNModelAve, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3)
        self.pool = nn.AvgPool2d(2, 2)
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = x.view(-1, 256) 
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
cnn_model_ave = CNNModelAve().to(device)
optimizer = optim.Adam(cnn_model_ave.parameters(), lr=0.001)

history_dict["Ave Pooling"] = train_model(cnn_model_ave, trainloader, testloader,
                                        loss_fn, optimizer, no_epochs)

### Strided Convolution

In [ ]:
def calculate_same_padding(kernel_size, stride):
    return (kernel_size - 1) // 2 if stride == 1 else ((stride - 1) + kernel_size - 1) // 2

class CNNModelStrided(nn.Module):
    def __init__(self):
        super(CNNModelStrided, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=calculate_same_padding(3, 2))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=calculate_same_padding(3, 2))
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=calculate_same_padding(3, 2))
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 4 * 4, 64)
        self.fc2 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
cnn_model_str = CNNModelStrided().to(device)
optimizer = optim.Adam(cnn_model_str.parameters(), lr=0.001)

history_dict["Strided Conv"] = train_model(cnn_model_str, trainloader, testloader,
                                        loss_fn, optimizer, no_epochs)

### Plot Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
def smoothed_points(points, factor=0.8):
    smoothed_points = []
    for point in points:
        if smoothed_points:
            previous = smoothed_points[-1]
            smoothed_points.append(previous * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

In [ ]:
fig, ax = plt.subplots(figsize=(14,10))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
for ilr in history_dict:
    val_loss_hist = history_dict[ilr]
    val_loss_values = val_loss_hist['test_acc']
    ax.plot(epochs, smoothed_points(val_loss_values), label = ilr, markevery=10)

ax.set_xlabel('Epochs')
ax.set_ylabel('Validation Accuracy')
ax.legend()
ax.set_ylim(50, 75)
fig.savefig('TestPooling.png', dpi=300, bbox_inches='tight')